# H15b: Explicit SVD Clamping vs. Muon
## Final-loss benchmark under post-step spectral controls

This notebook is the notebook counterpart to:
`experiments/Tier_1_Core_Mechanism_Tests/H15b_SVD_CLAMPING_VS_MUON/run_experiment.py`

It now uses the script's `run_experiment()` function as the **single computational source of truth** rather than duplicating the experiment loop.

### Scope of H15b
- This is a **controlled deep-linear toy benchmark** aimed at mechanism testing, not a broad performance study.
- The primary metric is **final loss after a fixed number of training steps**.
- The question is whether **conditioning-only post-step spectral controls** (SVD clamp / equalize) reproduce Muon's final-loss behavior under this setup.
- This notebook does **not** directly measure update-direction quality, generalization, or broader mechanistic sufficiency beyond this task / LR grid / threshold.
- "Best LR" below means **best on the tested LR grid using the designated sweep seeds**, not globally optimal.


In [ ]:
import importlib.util
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass

pd.options.display.float_format = lambda x: f"{x:.6g}"

EXPERIMENT_RELATIVE_PATH = Path(
    'experiments/Tier_1_Core_Mechanism_Tests/H15b_SVD_CLAMPING_VS_MUON/run_experiment.py'
)
KNOWN_PROJECT_ROOT = Path('/home/secemp9/Muon_as_RG_Gauge_Fixing')


def find_project_root_and_script():
    search_bases = []
    for base in [Path.cwd().resolve(), KNOWN_PROJECT_ROOT]:
        if base.exists() and base not in search_bases:
            search_bases.append(base)
            search_bases.extend(list(base.parents))
    seen = set()
    for root in search_bases:
        if root in seen:
            continue
        seen.add(root)
        script_path = root / EXPERIMENT_RELATIVE_PATH
        if script_path.exists():
            return root, script_path.parent, script_path
    raise FileNotFoundError(
        f'Could not locate {EXPERIMENT_RELATIVE_PATH} from cwd={Path.cwd().resolve()}'
    )


PROJECT_ROOT, EXPERIMENT_DIR, SCRIPT_PATH = find_project_root_and_script()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

spec = importlib.util.spec_from_file_location('h15b_svd_clamping_vs_muon', SCRIPT_PATH)
h15b = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = h15b
spec.loader.exec_module(h15b)

print(f'Project root:   {PROJECT_ROOT}')
print(f'Experiment dir: {EXPERIMENT_DIR}')
print(f'Script path:    {SCRIPT_PATH}')


## Reproducibility and runtime framing

This notebook preserves the script's default experiment unless a smoke flag is supplied via the environment.

- Full default run: same core setup as the script.
- Smoke run: reduced steps / seeds / LR grid for fast code-path verification.
- The full default run is computationally nontrivial because it performs many repeated 32x32 SVDs and Newton–Schulz projections.

The notebook records:
- configuration and seed schedule,
- LR-sweep results,
- chosen best-in-grid learning rates,
- per-seed final losses,
- convergence counts,
- final conditioning summaries from the trained models,
- T1/T2/T3 verdicts,
- actual loss-trajectory plots from the experiment.


In [ ]:
NOTEBOOK_SMOKE = os.environ.get('H15B_NOTEBOOK_SMOKE', '0') == '1'
CONFIG_OVERRIDES = h15b.SMOKE_CONFIG_OVERRIDES if NOTEBOOK_SMOKE else None
config = h15b.make_config(CONFIG_OVERRIDES)
workload = h15b.estimate_workload(config)
run_mode = 'SMOKE' if NOTEBOOK_SMOKE else 'FULL DEFAULT'

config_summary = {
    'run_mode': run_mode,
    'counterpart_script': str(SCRIPT_PATH),
    'dim': config['dim'],
    'num_layers': config['num_layers'],
    'num_steps': config['num_steps'],
    'num_seeds': config['num_seeds'],
    'batch_size': config['batch_size'],
    'momentum': config['momentum'],
    'ns_iters': config['ns_iters'],
    'lr_candidates': config['lr_candidates'],
    'kappa_targets': config['kappa_targets'],
    'lr_sweep_num_seeds': config['lr_sweep_num_seeds'],
    'divergence_threshold': config['divergence_threshold'],
    'seed_start': config['seed_start'],
    'seed_stride': config['seed_stride'],
}
workload_summary = {
    'estimated_total_train_runs': workload['total_train_runs'],
    'estimated_lr_sweep_train_runs': workload['lr_sweep_train_runs'],
    'estimated_full_phase_train_runs': workload['full_phase_train_runs'],
    'projection_calls_per_train_run': workload['projection_calls_per_train_run'],
    'estimated_total_svd_projection_calls': workload['total_svd_projection_calls'],
    'estimated_total_muon_projection_calls': workload['total_muon_projection_calls'],
}

print('Configuration summary')
display(pd.Series(config_summary, name='value').to_frame())
print('Workload estimate')
display(pd.Series(workload_summary, name='value').to_frame())
print('Seed schedule:', h15b.generate_seeds(config))
print(
    f"Best-in-grid LR selection uses the first {config['lr_sweep_num_seeds']} seeds only; "
    "this is a fixed grid search, not a global optimum claim."
)


## Execute the experiment

The next cell runs the script's `run_experiment()` function directly and stores the returned structured result dictionary. Loss histories are recorded so the notebook can show **actual experimental trajectories** rather than only endpoint summaries.


In [ ]:
start_wall = time.time()
results = h15b.run_experiment(
    config=CONFIG_OVERRIDES,
    verbose=False,
    record_full_phase_histories=True,
    record_conditioning=True,
)
wall_seconds = time.time() - start_wall

assert results['config']['num_steps'] == config['num_steps']
assert set(results['best_lrs']) == {entry['name'] for entry in results['method_configs']}

print(f"Notebook wall time: {wall_seconds:.2f} s")
print(f"Reported experiment runtime: {results['runtime_seconds']:.2f} s")
print('Result keys:', sorted(results.keys()))


## Final-loss summary by method

These are the main endpoint statistics for H15b. The primary quantity is mean **final** loss over finite runs. `best_lr_in_grid` means the best value found on the tested LR grid using the designated sweep seeds.


In [ ]:
summary_df = pd.DataFrame(results['summary_rows']).copy()
summary_df['ci95_halfwidth'] = 1.96 * summary_df['sem_final_loss']
summary_df = summary_df.sort_values('mean_final_loss').reset_index(drop=True)

summary_view = summary_df[
    [
        'label',
        'best_lr_in_grid',
        'mean_final_loss',
        'std_final_loss',
        'sem_final_loss',
        'ci95_halfwidth',
        'num_converged',
        'num_total',
        'ratio_vs_muon',
        'ratio_vs_sgd',
        'mean_final_product_kappa',
    ]
].rename(
    columns={
        'label': 'method',
        'best_lr_in_grid': 'best_lr_in_grid',
        'mean_final_loss': 'mean_final_loss',
        'std_final_loss': 'std_final_loss',
        'sem_final_loss': 'sem_final_loss',
        'ci95_halfwidth': 'approx_95pct_CI_halfwidth',
        'num_converged': 'converged',
        'num_total': 'total',
        'ratio_vs_muon': 'ratio_vs_muon',
        'ratio_vs_sgd': 'ratio_vs_sgd',
        'mean_final_product_kappa': 'mean_final_product_kappa',
    }
)
display(summary_view)

best_method = summary_df.iloc[0]
display(Markdown(
    f"**Reading guide.** Lower final loss is better. In this run, the lowest mean final loss is "
    f"**{best_method['label']}** at **{best_method['mean_final_loss']:.6e}**. "
    f"Uncertainty columns summarize variability across seeds, and convergence counts reveal whether "
    f"any runs diverged under the current threshold."
))


## Clamp-target comparison

This view isolates the SVD-clamp family and compares it against the SGD and Muon references. It is the cleanest way to ask whether progressively tighter explicit conditioning control closes the final-loss gap to Muon.


In [ ]:
clamp_df = summary_df[summary_df['method'] == 'sgd_clamp'].copy().sort_values('kappa_target')
muon_loss = float(summary_df.loc[summary_df['name'] == 'muon', 'mean_final_loss'].iloc[0])
sgd_loss = float(summary_df.loc[summary_df['name'] == 'sgd', 'mean_final_loss'].iloc[0])

x_vals = clamp_df['kappa_target'].astype(float).to_numpy()
y_vals = clamp_df['mean_final_loss'].astype(float).to_numpy()
sem_vals = clamp_df['sem_final_loss'].astype(float).to_numpy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_vals, y_vals, marker='o', linewidth=2, label='SGD + SVD clamp')
ax.fill_between(x_vals, y_vals - sem_vals, y_vals + sem_vals, alpha=0.2)
ax.axhline(muon_loss, color='tab:green', linestyle='--', linewidth=2, label='Muon reference')
ax.axhline(sgd_loss, color='tab:red', linestyle=':', linewidth=2, label='SGD reference')
ax.set_xscale('log')
ax.set_xlabel('Clamp target kappa')
ax.set_ylabel('Mean final loss')
ax.set_title('H15b clamp-target sweep: final loss vs explicit kappa target')
ax.legend()
plt.show()

display(
    clamp_df[
        ['label', 'kappa_target', 'best_lr_in_grid', 'mean_final_loss', 'std_final_loss', 'ratio_vs_muon']
    ].rename(columns={'label': 'method'})
)

display(Markdown(
    'This is still a final-loss comparison, not a direct measurement of direction quality. '
    'The question is whether stronger post-step conditioning control is sufficient to reproduce the Muon endpoint.'
))


## Actual loss trajectories from the experiment

The original H15b script only summarized endpoint losses. In this first completion pass, the script now exposes per-step loss histories for the **full-phase runs** so the notebook can show real training trajectories for representative methods.


In [ ]:
trajectory_methods = ['sgd', 'muon', 'sgd_clamp_k5', 'sgd_equalize']

fig, ax = plt.subplots(figsize=(9, 5.5))
for name in trajectory_methods:
    histories = [
        seed_result['loss_history']
        for seed_result in results['full_phase'][name]['seed_results']
        if seed_result['loss_history'] is not None
    ]
    history_array = np.asarray(histories, dtype=float)
    finite_rows = np.all(np.isfinite(history_array), axis=1)
    history_array = history_array[finite_rows]
    if history_array.size == 0:
        continue
    mean_history = history_array.mean(axis=0)
    sem_history = history_array.std(axis=0) / np.sqrt(history_array.shape[0])
    steps = np.arange(history_array.shape[1])
    label = summary_df.loc[summary_df['name'] == name, 'label'].iloc[0]
    ax.plot(steps, mean_history, linewidth=2, label=label)
    ax.fill_between(steps, mean_history - sem_history, mean_history + sem_history, alpha=0.15)

ax.set_xlabel('Recorded loss evaluation index')
ax.set_ylabel('Loss')
ax.set_yscale('log')
ax.set_title('Mean loss trajectories across finite full-phase runs')
ax.legend()
plt.show()

display(Markdown(
    "These curves are computed from the actual full-phase experiment runs at each method's selected best-in-grid LR. "
    "They support trajectory discussion inside this notebook, but they do not change the primary H15b endpoint, which remains final loss."
))


## Final conditioning diagnostics from the trained models

To support the conditioning narrative without redesigning the benchmark, the script now reports final per-layer and product condition numbers from the trained weights. These are measured on the **actual trained models**, not on a separate toy matrix demo.


In [ ]:
layer_kappa_cols = [f'layer_{i+1}_kappa' for i in range(config['num_layers'])]
layer_kappa_df = pd.DataFrame(summary_df['mean_final_layer_kappas'].tolist(), columns=layer_kappa_cols)
conditioning_df = pd.concat(
    [
        summary_df[['label', 'mean_final_product_kappa']].rename(columns={'label': 'method'}),
        layer_kappa_df,
    ],
    axis=1,
)
display(conditioning_df)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(conditioning_df['method'], conditioning_df['mean_final_product_kappa'])
ax.set_yscale('log')
ax.set_ylabel('Mean final product kappa (log scale)')
ax.set_title('Final product conditioning across methods')
ax.tick_params(axis='x', rotation=45)
plt.show()

display(Markdown(
    'Interpretation: clamp/equalize controls should strongly change these conditioning diagnostics if they are being applied correctly. '
    'Whether that conditioning improvement is sufficient to match Muon is a separate question answered by the final-loss tables and tests.'
))


## Learning-rate sweep details

The sweep data below make the LR-selection rule explicit. Each row is a candidate LR evaluated on the sweep seeds only. The chosen LR is simply the **minimum mean finite final loss on this grid**.


In [ ]:
lr_sweep_df = pd.DataFrame(results['lr_sweep_rows']).copy()
lr_sweep_view = lr_sweep_df[
    ['label', 'lr', 'mean_final_loss_finite', 'num_finite', 'num_total', 'selected_best_lr_in_grid']
].rename(
    columns={
        'label': 'method',
        'lr': 'lr',
        'mean_final_loss_finite': 'mean_final_loss_on_sweep_seeds',
        'num_finite': 'finite',
        'num_total': 'total',
        'selected_best_lr_in_grid': 'selected_best_lr',
    }
)
display(lr_sweep_view.sort_values(['method', 'lr'], ascending=[True, False]))

pivot = lr_sweep_df.pivot_table(index='label', columns='lr', values='mean_final_loss_finite')
print('Mean finite final loss on sweep seeds (rows=methods, cols=LR candidates)')
display(pivot)


## T1 / T2 / T3 verdicts

The benchmark's decision rules are kept explicit and scoped. These are statements about this setup's final-loss criterion, not blanket mechanistic proofs.


In [ ]:
tests_df = pd.DataFrame(
    [
        {
            'test': test_name,
            'pass': test_info['pass'],
            'ratio': test_info.get('ratio', np.nan),
            'threshold': test_info.get('threshold', np.nan),
            'description': test_info['description'],
            'scoped_interpretation': test_info['scoped_interpretation'],
        }
        for test_name, test_info in results['tests'].items()
    ]
)
display(tests_df)

for test_name in ['T1', 'T2', 'T3']:
    test_info = results['tests'][test_name]
    display(Markdown(
        f"**{test_name}.** {test_info['description']}  "
        f"**Verdict:** {'PASS' if test_info['pass'] else 'FAIL'}.  "
        f"{test_info['scoped_interpretation']}"
    ))


## Calibrated conclusion

The final statement below is intentionally narrow. It is generated from the actual computed results rather than from static theory prose.


In [ ]:
t1 = results['tests']['T1']['pass']
t2 = results['tests']['T2']['pass']
t3 = results['tests']['T3']['pass']
muon_row = summary_df.loc[summary_df['name'] == 'muon'].iloc[0]
sgd_row = summary_df.loc[summary_df['name'] == 'sgd'].iloc[0]
clamp5_row = summary_df.loc[summary_df['name'] == 'sgd_clamp_k5'].iloc[0]
eq_row = summary_df.loc[summary_df['name'] == 'sgd_equalize'].iloc[0]

if t3:
    headline = (
        "Under this setup, the tested conditioning-only post-step spectral controls do not reproduce Muon within the notebook's 2x criterion."
    )
else:
    headline = (
        "Under this setup, at least one conditioning-only post-step spectral control reaches the notebook's 2x final-loss criterion relative to Muon."
    )

conclusion_md = f"""
### Result-conditioned takeaway

**Headline:** {headline}

- Run mode: **{run_mode}**.
- Primary endpoint: mean **final loss after {config['num_steps']} steps**.
- Muon mean final loss: **{muon_row['mean_final_loss']:.6e}**.
- SGD mean final loss: **{sgd_row['mean_final_loss']:.6e}**.
- Clamp-at-5 mean final loss: **{clamp5_row['mean_final_loss']:.6e}** ({clamp5_row['ratio_vs_muon']:.2f}x Muon).
- Equalize mean final loss: **{eq_row['mean_final_loss']:.6e}** ({eq_row['ratio_vs_muon']:.2f}x Muon).

### What H15b supports
- The notebook now demonstrates the actual endpoint comparison, LR-selection rule, convergence counts, trajectories, and final conditioning diagnostics from the trained models.
- It supports statements about whether these **post-step spectral conditioning controls** did or did not reproduce Muon **in this benchmark**.

### What H15b does not establish by itself
- It does **not** directly measure update-direction quality.
- It does **not** prove a general mechanism outside this deep-linear setup.
- It does **not** justify calling the chosen LR globally optimal; the selection is only best-in-grid on the tested sweep seeds.
"""
display(Markdown(conclusion_md))
